# 5.1 Rumelhart Semantic Network

## Introduction

### Multilayer NN in Psychology: Semantics
  
Our minds are full of associations, presumably implemented as connection strengths between concepts. But associations can have a wide variety of structure -- they are not merely symmetric (e.g. "dog" and "mail carrier" are associated but have an asymmetric relationship -- the mail carrier is less likely to bite the dog than vice a versa). Semantic networks are graphs used to model the relationships between concepts in the mind, and are a commonly used representation of knowledge. They are depicted as graphs, whose vertices are concepts and whose edges represent connections between concepts. A concept that is a noun is connected to its characteristics, but also to categories that it is a member of. Nouns that share a category (e.g. birds and dogs are both animals) both have a connection to that category, and share a higher-order connection between that category and a category to which it belongs (e.g. animals belong to the category living things).  
  
In this way, semantic nets can contain a deductive logical hierarchy. One of the key features arising from this hierarchy is the "inheritance" property, which states that a concept belonging to a subcategory will inherit the properties of that subcategory and, by extension, any category to which the subcategory belongs.    
  
To clarify, consider hearing the phrase "Blue-Footed Booby" for the first time. Initially, it contains very little information. If you are then told that it is the name of a bird, you'll know that it must necessarily have feathers, a beak, feet, and eyes. You will also know that it is a living thing, and therefore grows and is comprised of cells. This is the inheritance property at work.  

![semantic net example](https://younesstrittmatter.github.io/502B/_static/images/sl_and_bp/semantic-example.png)  
  
  
Dave Rumelhart used neural networks to model semantic networks, recreating structured knowledge representations using hidden layers and backprop learning.  
  
In order to represent the semantic net computationally, Rumelhart came up with a particularly clever structure of a multi-layer network, with two input layers, two hidden layers, and four output layers.  
  
![semantic net architecture](https://younesstrittmatter.github.io/502B/_static/images/sl_and_bp/semantic-architecture.png) 
  
When training networks of this structure, it was found that the represenational nodes, when adequately trained, came to represent distinguishing features of their inputs. They were internal representations of the concepts similar to those created in the human mind.  
  
Let us explore his idea by building it ourselves.

**Installation and Setup**

In [3]:
%%capture
%pip install psyneulink

import numpy as np
import psyneulink as pnl
import matplotlib.pyplot as plt 

##  Rumelhart's Semantic Network 

The first thing we will do is create a data set with examples and labels that we can train the net on. We'll be using a similar data set to the one in Rumelhart's original paper on the semantic net [Rumelhart, Hinton, and Williams (1986)](https://princetonuniversity.github.io/NEU-PSY-502/_static/pdf/Class%209/Rumelhart1986.pdf).  

The "nouns" inputs are provided to you in a list, as are the "relations". The relations consist of "I", which is simply the list of nouns, "is" which is a list of largest categories to which the noun can belong, "has", which are potential physical attributes, and "can", which are abilities each noun possesses.  
  
The network takes as input a pair of nouns and relations, which form a priming statement, such as "daisy is a", and should produce a response from the network that includes both a list of what a daisy is, and lists of what a daisy has and can do, as well as the actual term "daisy". This response mimics the brain's priming response, by creating activation for concepts related to the one explicitly stated.  
  
This is a slightly simplified model of Rumelhart's network, which learned associations through internal structure only, not through explicit updating for every output.

In [4]:
# Stimuli and Relations

nouns = ['oak', 'pine', 'rose', 'daisy', 'canary', 'robin', 'salmon', 'sunfish']
relations = ['is', 'has', 'can']
is_list = ['living', 'living thing', 'plant', 'animal', 'tree', 'flower', 'bird', 'fish', 'big', 'green', 'red',
           'yellow']
has_list = ['roots', 'leaves', 'bark', 'branches', 'skin', 'feathers', 'wings', 'gills', 'scales']
can_list = ['grow', 'move', 'swim', 'fly', 'breathe', 'breathe underwater', 'breathe air', 'walk', 'photosynthesize']
descriptors = [nouns, is_list, has_list, can_list]

truth_nouns = np.identity(len(nouns))

truth_is = np.zeros((len(nouns), len(is_list)))

truth_is[0, :] = [1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]
truth_is[1, :] = [1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]
truth_is[2, :] = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
truth_is[3, :] = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0] # <--- Exercise 1a What does this row represent?
truth_is[4, :] = [1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1]
truth_is[5, :] = [1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1]
truth_is[6, :] = [1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0]
truth_is[7, :] = [1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]

truth_has = np.zeros((len(nouns), len(has_list)))

truth_has[0, :] = [1, 1, 1, 1, 0, 0, 0, 0, 0]
truth_has[1, :] = [1, 1, 1, 1, 0, 0, 0, 0, 0]
truth_has[2, :] = [1, 1, 0, 0, 0, 0, 0, 0, 0]
truth_has[3, :] = [1, 1, 0, 0, 0, 0, 0, 0, 0]
truth_has[4, :] = [0, 0, 0, 0, 1, 1, 1, 0, 0]
truth_has[5, :] = [0, 0, 0, 0, 1, 1, 1, 0, 0]
truth_has[6, :] = [0, 0, 0, 0, 0, 0, 0, 1, 1] # <--- Exercise 1b What does this row represent?
truth_has[7, :] = [0, 0, 0, 0, 0, 0, 0, 1, 1]

truth_can = np.zeros((len(nouns), len(can_list)))

truth_can[0, :] = [1, 0, 0, 0, 0, 0, 0, 0, 1]
truth_can[1, :] = [1, 0, 0, 0, 0, 0, 0, 0, 1] # <--- Exercise 1c What does this row represent?
truth_can[2, :] = [1, 0, 0, 0, 0, 0, 0, 0, 1]
truth_can[3, :] = [1, 0, 0, 0, 0, 0, 0, 0, 1]
truth_can[4, :] = [1, 1, 0, 1, 1, 0, 1, 1, 0]
truth_can[5, :] = [1, 1, 0, 1, 1, 0, 1, 1, 0]
truth_can[6, :] = [1, 1, 1, 0, 1, 1, 0, 0, 0]
truth_can[7, :] = [1, 1, 1, 0, 1, 1, 0, 0, 0]

truths = [[truth_nouns], [truth_is], [truth_has], [truth_can]]

#### Exercise 1{exercise}

In the above code, there are three rows that are marked with comments. For each of these rows, identify the noun that it represents, and explain what the row is representing.

Exercise 1a

Remember:
```python
nouns = ['oak', 'pine', 'rose', 'daisy', 'canary', 'robin', 'salmon', 'sunfish']
relations = ['is', 'has', 'can']
is_list = ['living', 'living thing', 'plant', 'animal', 'tree', 'flower', 'bird', 'fish', 'big', 'green', 'red',
           'yellow']
has_list = ['roots', 'leaves', 'bark', 'branches', 'skin', 'feathers', 'wings', 'gills', 'scales']
can_list = ['grow', 'move', 'swim', 'fly', 'breathe', 'breathe underwater', 'breathe air', 'walk', 'photosynthesize']
```

what does the row `truth_is[3, :]` represent?

```python
truth_is[3, :] = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
```

Solution 1a{solution}

The 3 in `truth_is[3, :]` refers to the 4th noun in the list of nouns (remember indexing in Python starts with 0), which is "daisy".

[1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0] can be interpreted as a list of 12 booleans, where 1 means true and 0 means false. We map this to the is_list, which is a list of 12 concepts.

That means:

- 1: "daisy" is a living thing
- 1: "daisy" is a living
- 1: "daisy" is a plant
- 0: "daisy" is not an animal
- 0: "daisy" is not a tree
- 1: "daisy" is a flower
- 0: "daisy" is not a bird
- 0: "daisy" is not a fish
- 0: "daisy" is not big
- 0: "daisy" is not green
- 0: "daisy" is not red
- 0: "daisy" is not yellow

Exercise 1b

Remember:
```python
nouns = ['oak', 'pine', 'rose', 'daisy', 'canary', 'robin', 'salmon', 'sunfish']
relations = ['is', 'has', 'can']
is_list = ['living', 'living thing', 'plant', 'animal', 'tree', 'flower', 'bird', 'fish', 'big', 'green', 'red',
           'yellow']
has_list = ['roots', 'leaves', 'bark', 'branches', 'skin', 'feathers', 'wings', 'gills', 'scales']
can_list = ['grow', 'move', 'swim', 'fly', 'breathe', 'breathe underwater', 'breathe air', 'walk', 'photosynthesize']
```

what does the row `truth_has[6, :]` represent?
```python
truth_has[6, :] = [0, 0, 0, 0, 0, 0, 0, 1, 1]
```

Solution 1b{solution}

- salmon does not have roots
- salmon does not have leaves
- salmon does not have a bark
- salmon does not have branches
- salmon does not have skin
- salmon does not have feathers
- salmon does not have wings
- salmon has gills
- salmon has scales

Exercise 1c

Remember:
```python
nouns = ['oak', 'pine', 'rose', 'daisy', 'canary', 'robin', 'salmon', 'sunfish']
relations = ['is', 'has', 'can']
is_list = ['living', 'living thing', 'plant', 'animal', 'tree', 'flower', 'bird', 'fish', 'big', 'green', 'red',
           'yellow']
has_list = ['roots', 'leaves', 'bark', 'branches', 'skin', 'feathers', 'wings', 'gills', 'scales']
can_list = ['grow', 'move', 'swim', 'fly', 'breathe', 'breathe underwater', 'breathe air', 'walk', 'photosynthesize']
```

what does the row `truth_can[1, :]` represent?
```python
truth_can[1, :] = [1, 0, 0, 0, 0, 0, 0, 0, 1]
```

Solution 1c{solution}

- pine can grow
- pine cannot move
- pine cannot swim
- pine cannot fly
- pine cannot breathe
- pine cannot breathe underwater
- pine cannot breathe air
- pine cannot walk
- pine can photosynthesize